# 🇵🇭 Philippine Address Parsing & Normalization Pipeline

**Architecture**

```
Raw addresses
     │
     ▼
Stage 1 ── Alias normalization      (ph_address_alias_extended_v4.csv)
     │
     ▼
Stage 2 ── PSGC fuzzy match          (rapidfuzz · province-anchored)
     │
     ▼
Stage 3 ── Confidence gate           (high → fast path · low → API)
     │                    │
     ▼                    ▼
  dim_location        Stage 4 ── Nominatim OSM API verify
  fast lookup              │
     │                    ▼
     └──────── Stage 5 ──────── Output split
                               │               │
                     verified_addresses.xlsx   invalid_addresses.xlsx
```

**Key design rules**
- A street/barangay token that *happens* to share a city name is **not** assigned that city unless a province or region corroborates it (fixes the *South Signal → General Santos* false match)
- Only ambiguous / low-confidence rows hit the Nominatim API — high-confidence rows are resolved purely from `dim_location` for speed
- 10 000 rows target: < 20 min total


## Cell 0 — Environment check
Verify all required packages are present before running the pipeline.

In [71]:
import importlib, sys

REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit the paths and thresholds here. Everything else in the pipeline reads from these variables.

| Variable | Default | Meaning |
|---|---|---|
| `CITY_SCORE_HIGH` | 88 | Auto-accept without API call |
| `CITY_SCORE_LOW` | 65 | Minimum to even consider a city match |
| `PROV_SCORE_HIGH` | 88 | Auto-accept province |
| `PROV_SCORE_LOW` | 60 | Minimum province score |
| `USE_API` | `True` | Set `False` for fuzzy-only mode (faster, less accurate) |
| `API_QUERY_MODE` | `'aggressive'` | API search mode: `'strict'` = minimal queries, `'aggressive'` = typo-tolerant fallbacks |
| `MAX_ROWS` | `None` | Set an integer (e.g. `100`) to process only a slice for testing |


In [105]:
from pathlib import Path

# ── File paths ──────────────
BASE_DIR = Path("..") / "data"   # notebook is in update_address/notebooks

INPUT_FILE   = str(BASE_DIR / "sample"  / "sample_org_address.xlsx")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
OUT_VERIFIED  = str(BASE_DIR / "output" / "verified_addresses.xlsx")
OUT_INVALID   = str(BASE_DIR / "output" / "invalid_addresses.xlsx")

# ── Fuzzy-match thresholds (0-100) ───────────────────────────────────────────
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# ── Nominatim (OSM) API settings ─────────────────────────────────────────────
NOMINATIM_URL = 'https://nominatim.openstreetmap.org/search'
NOMINATIM_UA  = 'ph-address-pipeline/1.0 (research)'  # OSM requires a User-Agent
BATCH_DELAY   = 1.1    # seconds between requests — OSM policy: max 1 req/sec
MAX_RETRIES   = 2

# ── Run controls ─────────────────────────────────────────────────────────────
USE_API         = True          # False = fuzzy-only
API_QUERY_MODE  = 'aggressive'  # 'strict' | 'aggressive'
MAX_ROWS        = None          # None = all rows; set e.g. 100 for a quick test run

# ── Quick path check ─────────────────────────────────────────────────────────
for f in [INPUT_FILE, DIM_LOCATION, ALIAS_FILE]:
    status = '✅' if Path(f).exists() else '❌  NOT FOUND'
    print(f'  {status}  {f}')
print(f'  ℹ️  API_QUERY_MODE = {API_QUERY_MODE}')


  ✅  ..\data\sample\sample_org_address.xlsx
  ✅  ..\data\mapping\dim_location_20260316_v2.csv
  ✅  ..\data\utils\ph_address_alias_extended_v4.csv
  ℹ️  API_QUERY_MODE = aggressive


## Cell 2 — Imports & logging setup

In [73]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


17:56:55  INFO      Imports OK


## Cell 3 — Load reference data

Loads `dim_location` (42 000+ PSGC barangay rows) and `ph_address_alias` (499 alias pairs), then builds fast in-memory lookup lists for fuzzy matching.

In [74]:
def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = pd.read_csv(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).str.upper().str.strip()

    log.info('Loading alias table ...')
    alias = pd.read_csv(alias_path)
    alias['pattern']     = alias['pattern'].fillna('').astype(str).str.upper().str.strip()
    alias['replacement'] = alias['replacement'].fillna('').astype(str).str.upper().str.strip()

    cities    = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions   = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n── Sample dim_location ──')
display(dim.head(3))
print('\n── Sample aliases ──')
display(alias_df.head(10))


17:56:55  INFO      Loading dim_location ...
17:56:55  INFO      Loading alias table ...
17:56:55  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



── Sample dim_location ──


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



── Sample aliases ──


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


## Cell 4 — Stage 1: Alias normalization

Builds a **single-pass compiled regex** from all 499 alias pairs (sorted longest-first so multi-word patterns like `CITY OF MARIKINA` fire before the single-token `MARIKINA`).

A post-pass deduplication step catches chained-alias artifacts like `CITY CITY` or `BARANGAY BARANGAY` that arise when both `CITY OF X → X CITY` and `X → X CITY` fire on the same string.

In [75]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest pattern first
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = str(text).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)
    # Remove duplicate consecutive words (e.g. 'CITY CITY', 'BARANGAY BARANGAY')
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

# ── Quick test on sample addresses ───────────────────────────────────────────
test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('─' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')


17:56:55  INFO      Alias regex built from 499 patterns


ORIGINAL                                                 NORMALIZED
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY (CAPITAL) MISAMIS ORIENTAL


## Cell 5 — Stage 2: Fuzzy matching helpers

Two functions used throughout the pipeline:
- **`best_match`** — matches a single query string against a list (full-string scorer)
- **`token_scan`** — splits the address into tokens and tries each token independently; returns the highest-scoring hit across all tokens

Both use `rapidfuzz.fuzz.WRatio` which handles partial overlaps well (e.g. `QUEZON CITY` inside a longer address string).

In [76]:
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.WRatio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Return (best_match_string, score) or (None, 0) if below cutoff."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Try each token against choices; return best (match, score)."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Quick smoke test ──────────────────────────────────────────────────────────
tests = [
    ('QUEZON CITY', cities),
    ('ILOCOS SUR', provinces),
    ('TAGUIG', cities),
    ('SOUTH SIGNAL', cities),   # should score low / return nothing meaningful
]
print(f'{"Query":<25}  {"Best match":<35}  Score')
print('─' * 75)
for q, lst in tests:
    m, s = best_match(q, lst, score_cutoff=0)
    print(f'{q:<25}  {str(m):<35}  {s}')


Query                      Best match                           Score
───────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                          100
ILOCOS SUR                 ILOCOS SUR                           100
TAGUIG                     CITY OF TAGUIG                       90
SOUTH SIGNAL               SIGAY                                72


## Cell 6 — Stage 2 + 3: Core address matching & confidence gate

### Strict city-detection rule
A token scoring ≥ `CITY_SCORE_LOW` is a **candidate**, not a result.  
If that candidate city exists in multiple provinces (ambiguous), it **requires** the province to also appear in the address with score ≥ `PROV_SCORE_LOW` before the city is assigned.  
This is what prevents `South Signal Village` from being mapped to `City of General Santos` — `SIGNAL` has no city match, and the correct `TAGUIG` token maps cleanly.

### Confidence levels
| Level | Condition | Action |
|---|---|---|
| `high` | city score ≥ 88 | Accept immediately — no API |
| `medium` | city score 65–87 | Send to Nominatim for verification |
| `low` | no city, province score < 88 | Send to Nominatim |
| `none` | nothing matched | Mark invalid |


In [77]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
    )

    tokens = [t for t in re.split(r'[,\s]+', address_norm) if len(t) >= 3]

    # ── Build context hints first (used for robust barangay disambiguation) ─────
    prov_hint, prov_hint_score = token_scan(tokens, provinces, score_cutoff=prov_low)
    city_full_hint, city_full_score = best_match(address_norm, cities, score_cutoff=city_low)
    city_tok_hint, city_tok_score = token_scan(tokens, cities, score_cutoff=city_low)
    city_hint, city_hint_score = (
        (city_full_hint, city_full_score)
        if city_full_score >= city_tok_score
        else (city_tok_hint, city_tok_score)
    )

    def _name_overlaps_address(name: str, text: str) -> bool:
        skip = {'CITY', 'MUNICIPALITY', 'OF', 'PROVINCE'}
        words = [w for w in str(name).split() if w not in skip and len(w) >= 4]
        return any(w in text for w in words)

    generic_brgy_terms = {
        'PUROK', 'POBLACION', 'ZONE', 'CENTRO', 'PROPER',
        'BARANGAY', 'SITIO', 'VILLA', 'DISTRICT', 'AREA'
    }

    def _best_barangay_row(brgy_name: Optional[str], brgy_raw_score: int, source: str):
        if not brgy_name:
            return None, -1, -1
        subset = dim[dim['barangay_name'] == brgy_name]
        if subset.empty:
            return None, -1, -1

        best_row = None
        best_adjusted = -1
        best_bonus = -1
        for _, row in subset.iterrows():
            bonus = 0
            if city_hint and row['city_name'] == city_hint:
                bonus += 25
            if prov_hint and row['province_name'] == prov_hint:
                bonus += 20
            if _name_overlaps_address(row['city_name'], address_norm):
                bonus += 10
            if _name_overlaps_address(row['province_name'], address_norm):
                bonus += 8

            adjusted = brgy_raw_score + bonus
            if adjusted > best_adjusted:
                best_adjusted = adjusted
                best_bonus = bonus
                best_row = row

        # Accept barangay anchor only when context supports it.
        # Prevents generic-token false positives like PUROK/BARANG/LUZON/etc.
        brgy_upper = str(brgy_name).upper().strip()
        brgy_words = [w for w in brgy_upper.split() if w]
        is_generic = brgy_upper in generic_brgy_terms
        is_short_single = len(brgy_words) == 1 and len(brgy_upper) < 7
        looks_placeholder = brgy_upper.startswith('BARANG')

        if source == 'token' and best_bonus < 10:
            return None, -1, -1
        if is_generic and best_bonus < 15:
            return None, -1, -1
        if (is_short_single or looks_placeholder) and best_bonus < 20:
            return None, -1, -1
        if len(brgy_words) == 1 and source == 'full' and best_bonus < 10:
            return None, -1, -1

        return best_row, best_adjusted, best_bonus

    # ── PRIORITY 1: Barangay-first with context-aware candidate scoring ─────────
    # Generic behavior: handles all addresses where a misleading token competes
    # with a true multi-word barangay (e.g., LUZON vs BATASAN HILLS).
    brgy_full, brgy_score_full = best_match(address_norm, barangays, score_cutoff=BRGY_SCORE_MIN)
    brgy_tok,  brgy_score_tok  = token_scan(tokens, barangays, score_cutoff=BRGY_SCORE_MIN)

    row_full, adj_full, bonus_full = _best_barangay_row(brgy_full, brgy_score_full, source='full')
    row_tok,  adj_tok,  bonus_tok  = _best_barangay_row(brgy_tok, brgy_score_tok, source='token')

    if max(adj_full, adj_tok) >= BRGY_SCORE_MIN:
        brgy_row = row_full if adj_full >= adj_tok else row_tok
        result.update(
            city_name=brgy_row['city_name'],
            city_code=int(brgy_row['city_code']),
            city_score=95,
            province_name=brgy_row['province_name'],
            province_code=int(brgy_row['province_code']),
            province_score=95,
            region_name=brgy_row['region_name'],
            region_code=int(brgy_row['region_code']),
            barangay_name=brgy_row['barangay_name'],
            barangay_code=int(brgy_row['barangay_code']),
            psgc_10_digit=int(brgy_row['psgc_10_digit']),
            confidence='high', needs_api=False,
        )
        return result

    # ── PRIORITY 2: Fall back to province + city matching if no barangay ──────
    prov_match, prov_score = prov_hint, prov_hint_score

    # ── City scan (full string + token scan; take best) ───────────────────
    city_match, city_score = city_hint, city_hint_score

    # ── Strict disambiguation ─────────────────────────────────────────────
    # Always require province corroboration for cities that exist in >1 province.
    if city_match:
        candidate_provs = dim[dim['city_name'] == city_match]['province_name'].unique()
        if len(candidate_provs) > 1:
            if not (prov_match and prov_match in candidate_provs
                    and prov_score >= prov_low):
                city_score = 0
                city_match = None

    # ── Populate result from dim_location ────────────────────────────────
    if city_match:
        subset = dim[dim['city_name'] == city_match]
        if prov_match and prov_score >= prov_low:
            sub2 = subset[subset['province_name'] == prov_match]
            if not sub2.empty:
                subset = sub2
        first = subset.iloc[0]
        result.update(
            city_name=city_match, city_code=int(first['city_code']),
            city_score=city_score,
            province_name=first['province_name'], province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )
    elif prov_match and prov_score >= prov_high:
        subset = dim[dim['province_name'] == prov_match]
        first = subset.iloc[0]
        result.update(
            province_name=prov_match, province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )

    # ── Confidence classification ─────────────────────────────────────────
    cs = result['city_score']
    ps = result['province_score']
    if cs >= city_high:
        result['confidence'], result['needs_api'] = 'high', False
    elif cs >= city_low:
        result['confidence'], result['needs_api'] = 'medium', True
    elif result['province_name'] and ps >= prov_high:
        result['confidence'], result['needs_api'] = 'medium', True
    else:
        result['confidence'], result['needs_api'] = 'low', True

    return result


# ── Generalized regression tests (not single-case tuning) ────────────────────
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
]

for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print('-' * 80)


Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Barangay   : BATASAN HILLS
  City       : QUEZON CITY
  Province   : METRO MANILA
  Confidence : high
--------------------------------------------------------------------------------
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
  Barangay   : SOUTH SIGNAL VILLAGE
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Confidence : high
--------------------------------------------------------------------------------


## Cell 7 — Run Stage 1–3 on all input rows

Loads the input file, applies alias normalization, then runs the fuzzy matcher on every row. Results are split into `high_conf` (no API needed) and `low_conf` (needs Nominatim verification).

In [98]:
import time

t_start = time.time()

log.info(f'Loading {INPUT_FILE} ...')
df_input = pd.read_excel(INPUT_FILE)
if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# ── Stage 1: alias normalization ──────────────────────────────────────────────
log.info('Stage 1: alias normalization ...')
df_input['address_normalized'] = df_input[addr_col].apply(
    lambda x: normalize_alias(str(x), compiled_re, replace_map)
)

# ── Stage 2-3: fuzzy match + confidence gate ──────────────────────────────────
log.info('Stage 2-3: fuzzy matching + confidence gate ...')
records = []

for _, row in tqdm(df_input.iterrows(), total=len(df_input),
                   desc='Fuzzy match', unit='row'):
    rec = {
        'original_address'  : row[addr_col],
        'address_normalized': row['address_normalized'],
        'api_status': 'pending' if USE_API else 'skipped',
        'osm_city': None, 'osm_province': None, 'osm_region': None,
        'osm_display': None, 'osm_lat': None, 'osm_lon': None,
    }
    rec.update(match_address(
        row['address_normalized'],
        dim, cities, provinces, regions, barangays,
    ))
    records.append(rec)

# When USE_API=True, all rows are queued for API double-check.
if USE_API:
    high_conf = []
    low_conf = records
else:
    high_conf = [r for r in records if not r['needs_api']]
    low_conf  = [r for r in records if r['needs_api']]

t_fuzzy = time.time() - t_start
log.info(f'Fuzzy pass done in {t_fuzzy:.1f}s')
if USE_API:
    log.info(f'  Queued for API verification : {len(low_conf):,} (all rows)')
else:
    log.info(f'  High confidence (no API): {len(high_conf):,}')
    log.info(f'  Needs API verification  : {len(low_conf):,}')

# ── Preview ───────────────────────────────────────────────────────────────────
df_preview = pd.DataFrame(records)[[
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'city_score'
]]
display(df_preview)


18:08:46  INFO      Loading ..\data\sample\sample_org_address.xlsx ...
18:08:46  INFO      Address column: "Original_Address"  |  Total rows: 7
18:08:46  INFO      Stage 1: alias normalization ...
18:08:46  INFO      Stage 2-3: fuzzy matching + confidence gate ...


Fuzzy match:   0%|          | 0/7 [00:00<?, ?row/s]

18:08:48  INFO      Fuzzy pass done in 2.0s
18:08:48  INFO        Queued for API verification : 7 (all rows)


,original_address,address_normalized,city_name,province_name,confidence,city_score
0,19 Luzon Street Batasan Hills 67C Jacson St Qu...,19 LUZON STREET BATASAN HILLS 67C JACSON STREE...,QUEZON CITY,METRO MANILA,high,95
1,sapang palay sjdm,SAPANG PALAY SAN JOSE DEL MONTE CITY,NaN,AGUSAN DEL NORTE,medium,0
2,12 General Espino South Signal,12 GENERAL ESPINO SOUTH SIGNAL,CITY OF GENERAL SANTOS,SOUTH COTABATO,high,90
3,loilo City,LOILO CITY,CITY OF ILOILO,ILOILO,high,90
4,55 Dawn St South Kim View Park SSS Village Con...,55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE...,CITY OF KORONADAL,SOUTH COTABATO,high,95
5,Vigan ilocos sur,VIGAN CITY ILOCOS SUR,VIGA,CATANDUANES,high,90
6,carmencdoc Carmen CAGAYAN DE ORO CITY (Capital...,CARMENCDOC CARMEN CAGAYAN DE ORO CITY (CAPITAL...,CARMEN,SURIGAO DEL SUR,high,95


In [99]:
# Snapshot fuzzy outputs before API mutates records
pre_api_df = pd.DataFrame([dict(r) for r in records])
print(f'Captured pre_api_df rows: {len(pre_api_df):,}')

Captured pre_api_df rows: 7


## Cell 8 — Stage 4: Nominatim API verification

When `USE_API=True`, **all rows** are sent to the [OpenStreetMap Nominatim API](https://nominatim.org/) for double-checking.  
The API result is re-matched against `dim_location` and becomes the final location source.

If no reliable city/province can be derived from API after retries and alternate query forms, the row is marked **unverified** and will become invalid in Stage 5.

`API_QUERY_MODE` controls how broad fallback searching is:
- `'strict'` → original + normalized queries only
- `'aggressive'` → typo-tolerant cleanup + city-hint candidate queries

**OSM usage policy:** 1 request per second maximum. `BATCH_DELAY = 1.1s` keeps us within policy.  
**Set `USE_API = False` in Cell 1** to run fuzzy-only mode.

> 💡 **Estimated API time:** `row_count × 1.1s` (plus retries when needed).


In [106]:
def _nominatim_query(address: str) -> Optional[dict]:
    params = urllib.parse.urlencode({
        'q': address + ', Philippines',
        'format': 'json',
        'addressdetails': 1,
        'limit': 1,
        'countrycodes': 'ph',
    })
    req = urllib.request.Request(
        f'{NOMINATIM_URL}?{params}',
        headers={'User-Agent': NOMINATIM_UA}
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
        return data[0] if data else None
    except Exception as e:
        log.debug(f'Nominatim error: {e}')
        return None


def _osm_extract(result: dict) -> dict:
    addr = result.get('address', {})
    city_raw = (
        addr.get('city') or addr.get('town') or
        addr.get('municipality') or addr.get('county') or ''
    )
    return {
        'osm_city':    city_raw.upper().strip(),
        'osm_province': (addr.get('state_district') or addr.get('province') or '').upper().strip(),
        'osm_region':   (addr.get('state') or '').upper().strip(),
        'osm_lat':      result.get('lat'),
        'osm_lon':      result.get('lon'),
        'osm_display':  result.get('display_name', ''),
    }


def _clear_location_fields(row: dict):
    row.update(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
    )


def _candidate_queries(row: dict, cities: list, query_mode: str = 'aggressive') -> list:
    original = str(row.get('original_address') or '').strip()
    normalized = str(row.get('address_normalized') or '').strip()

    candidates = []

    def _add(q: str):
        q = str(q or '').strip(' ,')
        if q and q not in candidates:
            candidates.append(q)

    # Baseline candidates used by both modes
    _add(original)
    _add(normalized)

    if query_mode == 'strict':
        return candidates[:6]

    # Aggressive mode: typo/noise-tolerant fallbacks
    cleaned = re.sub(r'[^A-Z0-9\s,]', ' ', normalized.upper())
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    _add(cleaned)

    no_house_num = re.sub(r'\b\d+[A-Z]?\b', ' ', cleaned)
    no_house_num = re.sub(r'\s+', ' ', no_house_num).strip()
    _add(no_house_num)

    # Extract explicit city phrases like "MARIKINA CITY", "QUEZON CITY"
    for m in re.findall(r'\b([A-Z]+(?:\s+[A-Z]+){0,4}\sCITY)\b', cleaned):
        _add(m)

    # Add fuzzy city hint from normalized string for typo tolerance
    city_hint, city_score = best_match(cleaned, cities, score_cutoff=82)
    if city_hint:
        _add(city_hint)

    # Combine city-like hints with province hint already carried by fuzzy stage
    if row.get('province_name'):
        for q in candidates[:]:
            if q.endswith('CITY') or q.startswith('CITY OF'):
                _add(f'{q}, {row.get("province_name")}')

    # Last-resort short query from last 4-6 tokens (often contains city/province)
    toks = [t for t in re.split(r'[\s,]+', cleaned) if t]
    if len(toks) >= 4:
        _add(' '.join(toks[-6:]))
        _add(' '.join(toks[-4:]))

    return candidates[:12]


def _query_with_fallbacks(row: dict, cities: list, query_mode: str = 'aggressive') -> Optional[dict]:
    query_candidates = _candidate_queries(row, cities, query_mode=query_mode)

    for q in query_candidates:
        for _ in range(MAX_RETRIES + 1):
            osm_raw = _nominatim_query(q)
            if osm_raw:
                return osm_raw
            time.sleep(BATCH_DELAY)
    return None


def api_verify_batch(
    rows: list, dim: pd.DataFrame,
    cities: list, provinces: list, regions: list,
    query_mode: str = 'aggressive',
) -> list:
    mode = str(query_mode).strip().lower()
    if mode not in {'strict', 'aggressive'}:
        mode = 'aggressive'
    log.info(f'Sending {len(rows)} rows to Nominatim ... (mode={mode})')
    out = []
    for row in tqdm(rows, desc='API verify', unit='addr'):
        osm_raw = _query_with_fallbacks(row, cities, query_mode=mode)

        if not osm_raw:
            _clear_location_fields(row)
            row.update(confidence='none', api_status='no_result', needs_api=False)
            out.append(row)
            time.sleep(BATCH_DELAY)
            continue

        osm = _osm_extract(osm_raw)
        row.update(osm)

        c_match, c_score = (None, 0)
        p_match, p_score = (None, 0)

        if osm['osm_city']:
            c_match, c_score = best_match(osm['osm_city'], cities, score_cutoff=CITY_SCORE_LOW)
        if osm['osm_province']:
            p_match, p_score = best_match(osm['osm_province'], provinces, score_cutoff=PROV_SCORE_LOW)

        matched = False

        if c_match and c_score >= CITY_SCORE_LOW:
            subset = dim[dim['city_name'] == c_match]

            candidate_provs = subset['province_name'].unique()
            if len(candidate_provs) > 1:
                if p_match and p_match in candidate_provs and p_score >= PROV_SCORE_LOW:
                    sub2 = subset[subset['province_name'] == p_match]
                    if not sub2.empty:
                        subset = sub2
                    else:
                        subset = subset.iloc[0:0]
                else:
                    subset = subset.iloc[0:0]
            elif p_match and p_score >= PROV_SCORE_LOW:
                sub2 = subset[subset['province_name'] == p_match]
                if not sub2.empty:
                    subset = sub2

            if not subset.empty:
                first = subset.iloc[0]
                row.update(
                    city_name=c_match, city_code=int(first['city_code']), city_score=int(c_score),
                    province_name=first['province_name'], province_code=int(first['province_code']),
                    province_score=max(int(p_score), PROV_SCORE_LOW) if p_match else 0,
                    region_name=first['region_name'], region_code=int(first['region_code']),
                    confidence='api_verified', api_status='matched', needs_api=False,
                )

                city_brgy = dim[dim['city_name'] == c_match]['barangay_name'].tolist()
                tokens = [t for t in re.split(r'[,\s]+', row.get('address_normalized', '')) if len(t) >= 3]
                brgy_match, brgy_score = token_scan(tokens, city_brgy, score_cutoff=BRGY_SCORE_MIN)
                if brgy_match:
                    brgy_row = dim[
                        (dim['city_name'] == c_match) &
                        (dim['barangay_name'] == brgy_match)
                    ].iloc[0]
                    row.update(
                        barangay_name=brgy_match,
                        barangay_code=int(brgy_row['barangay_code']),
                        psgc_10_digit=int(brgy_row['psgc_10_digit']),
                    )
                matched = True

        if not matched and p_match and p_score >= PROV_SCORE_LOW:
            subset = dim[dim['province_name'] == p_match]
            if not subset.empty:
                first = subset.iloc[0]
                row.update(
                    city_name=None, city_code=None, city_score=0,
                    province_name=p_match, province_code=int(first['province_code']),
                    province_score=int(p_score),
                    region_name=first['region_name'], region_code=int(first['region_code']),
                    barangay_name=None, barangay_code=None, psgc_10_digit=None,
                    confidence='api_verified_province', api_status='province_only', needs_api=False,
                )
                matched = True

        if not matched:
            _clear_location_fields(row)
            row.update(confidence='none', api_status='unmatched', needs_api=False)

        out.append(row)
        time.sleep(BATCH_DELAY)

    return out


# ── Run API stage ─────────────────────────────────────────────────────────────
if USE_API and low_conf:
    t_api_start = time.time()
    low_conf = api_verify_batch(
        low_conf, dim, cities, provinces, regions,
        query_mode=API_QUERY_MODE,
    )
    log.info(f'API stage done in {time.time() - t_api_start:.1f}s')
else:
    if not USE_API:
        log.info('USE_API=False — skipping Nominatim stage')
    else:
        log.info('No rows available for API verification')


18:12:32  INFO      Sending 7 rows to Nominatim ... (mode=aggressive)


API verify:   0%|          | 0/7 [00:00<?, ?addr/s]

18:13:40  INFO      API stage done in 67.1s


## Cell 9 — Stage 5a: Merge results & determine validity

Combines the processed rows and applies strict validity rules.

- **When `USE_API=True`**: valid only if API verification resolved city/province (`api_status` in `matched`, `province_only`)
- **When `USE_API=False`**: fallback fuzzy-only validity logic is used

This ensures rows with unresolved location after API retries are marked invalid.

In [101]:
all_records = low_conf if USE_API else (high_conf + low_conf)
out_df = pd.DataFrame(all_records)

# Final validity flag
# - API mode: valid only when API verification produced city/province mapping
# - Fuzzy mode: keep previous rule
if USE_API:
    out_df['is_valid'] = (
        out_df['api_status'].isin(['matched', 'province_only']) &
        (
            out_df['city_name'].notna() |
            out_df['province_name'].notna()
        )
    )
else:
    out_df['is_valid'] = (
        out_df['city_name'].notna() |
        (
            out_df['province_name'].notna() &
            (out_df['province_score'] >= PROV_SCORE_HIGH)
        )
    )

# Confidence value counts
print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'api_status' in out_df.columns:
    print('\nAPI status distribution:')
    print(out_df['api_status'].value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


Confidence distribution:
confidence
api_verified    7

API status distribution:
api_status
matched    7

Valid   : 7
Invalid : 0
Total   : 7


In [102]:
# Before-vs-after change report (fuzzy baseline vs API-final)
compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']
base = pre_api_df[['original_address'] + compare_cols].copy()
base = base.rename(columns={
    'city_name': 'city_before',
    'province_name': 'province_before',
    'confidence': 'confidence_before',
    'api_status': 'api_status_before',
})

final = out_df[['original_address'] + compare_cols].copy()
final = final.rename(columns={
    'city_name': 'city_after',
    'province_name': 'province_after',
    'confidence': 'confidence_after',
    'api_status': 'api_status_after',
})

change_df = base.merge(final, on='original_address', how='inner')
changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)

changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)

Rows changed after API verification: 7 / 7


,original_address,city_before,province_before,confidence_before,api_status_before,city_after,province_after,confidence_after,api_status_after
0,19 Luzon Street Batasan Hills 67C Jacson St Qu...,QUEZON CITY,METRO MANILA,high,pending,QUEZON CITY,METRO MANILA,api_verified,matched
1,sapang palay sjdm,NaN,AGUSAN DEL NORTE,medium,pending,CITY OF SAN JOSE DEL MONTE,BULACAN,api_verified,matched
2,12 General Espino South Signal,CITY OF GENERAL SANTOS,SOUTH COTABATO,high,pending,CITY OF TAGUIG,METRO MANILA,api_verified,matched
3,loilo City,CITY OF ILOILO,ILOILO,high,pending,ALAMINOS,LAGUNA,api_verified,matched
4,55 Dawn St South Kim View Park SSS Village Con...,CITY OF KORONADAL,SOUTH COTABATO,high,pending,CITY OF MARIKINA,METRO MANILA,api_verified,matched
5,Vigan ilocos sur,VIGA,CATANDUANES,high,pending,CITY OF VIGAN,ILOCOS SUR,api_verified,matched
6,carmencdoc Carmen CAGAYAN DE ORO CITY (Capital...,CARMEN,SURIGAO DEL SUR,high,pending,ABRA DE ILOG,OCCIDENTAL MINDORO,api_verified,matched


In [103]:
# Compact delta view
compact = changed_rows[[
    'original_address',
    'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

# Highlight the user's example row
mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address',
        'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])

,original_address,city_before,city_after,province_before,province_after,api_status_before,api_status_after
0,19 Luzon Street Batasan Hills 67C Jacson St Qu...,QUEZON CITY,QUEZON CITY,METRO MANILA,METRO MANILA,pending,matched
1,sapang palay sjdm,NaN,CITY OF SAN JOSE DEL MONTE,AGUSAN DEL NORTE,BULACAN,pending,matched
2,12 General Espino South Signal,CITY OF GENERAL SANTOS,CITY OF TAGUIG,SOUTH COTABATO,METRO MANILA,pending,matched
3,loilo City,CITY OF ILOILO,ALAMINOS,ILOILO,LAGUNA,pending,matched
4,55 Dawn St South Kim View Park SSS Village Con...,CITY OF KORONADAL,CITY OF MARIKINA,SOUTH COTABATO,METRO MANILA,pending,matched
5,Vigan ilocos sur,VIGA,CITY OF VIGAN,CATANDUANES,ILOCOS SUR,pending,matched
6,carmencdoc Carmen CAGAYAN DE ORO CITY (Capital...,CARMEN,ABRA DE ILOG,SURIGAO DEL SUR,OCCIDENTAL MINDORO,pending,matched


General Espino South Signal change:


,original_address,city_before,city_after,province_before,province_after,confidence_before,confidence_after,api_status_before,api_status_after
2,12 General Espino South Signal,CITY OF GENERAL SANTOS,CITY OF TAGUIG,SOUTH COTABATO,METRO MANILA,high,api_verified,pending,matched


## Cell 10 — Stage 5b: Export to Excel

Writes two styled `.xlsx` files:
- **`verified_addresses.xlsx`** — green header tab, all resolved PSGC fields
- **`invalid_addresses.xlsx`** — red header tab, original + normalized address + debug info

Both files have frozen header rows and auto-fitted column widths.

In [104]:
VERIFIED_COLS = [
    'original_address', 'address_normalized',
    'barangay_name', 'barangay_code',
    'city_name', 'city_code',
    'province_name', 'province_code',
    'region_name', 'region_code',
    'psgc_10_digit',
    'confidence', 'city_score', 'province_score',
    'osm_display', 'osm_lat', 'osm_lon',
]
INVALID_COLS = [
    'original_address', 'address_normalized',
    'confidence', 'city_score', 'province_score',
    'osm_display', 'api_status',
]

verified_df = out_df[out_df['is_valid']][
    [c for c in VERIFIED_COLS if c in out_df.columns]
].reset_index(drop=True)

invalid_df = out_df[~out_df['is_valid']][
    [c for c in INVALID_COLS if c in out_df.columns]
].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows → {path}')


write_xlsx(verified_df, OUT_VERIFIED, 'Verified', '#00B050')
write_xlsx(invalid_df,  OUT_INVALID,  'Invalid',  '#FF0000', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print(f'Output files:')
print(f'  ✅  {OUT_VERIFIED}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {OUT_INVALID}   ({len(invalid_df):,} rows)')


18:10:07  INFO      Wrote 7 rows → ..\data\output\verified_addresses.xlsx
18:10:07  INFO      Wrote 0 rows → ..\data\output\invalid_addresses.xlsx



Total elapsed: 81.0s  (1.35 min)
Output files:
  ✅  ..\data\output\verified_addresses.xlsx  (7 rows)
  ⚠️   ..\data\output\invalid_addresses.xlsx   (0 rows)


## Cell 11 — Results summary

Quick inline preview of both output tables for a final sanity check.

In [85]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<20} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf): {high_c["city_score"].mean():.1f}')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


════════════════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Input rows        :        7
  Verified          :        3  (42.9%)
  Invalid           :        4  (57.1%)

  Confidence breakdown:
    none                      4  (57.1%)
    api_verified              3  (42.9%)

  Total elapsed     : 42.2s
════════════════════════════════════════════════════════════════════════════════

── Verified (first 10 rows) ──


,original_address,address_normalized,barangay_name,barangay_code,city_name,city_code,province_name,province_code,region_name,region_code,psgc_10_digit,confidence,city_score,province_score,osm_display,osm_lat,osm_lon
0,sapang palay sjdm,SAPANG PALAY SAN JOSE DEL MONTE CITY,SAPANG PALAY,10.0,CITY OF SAN JOSE DEL MONTE,20.0,BULACAN,14.0,REGION III (CENTRAL LUZON),3.0,3.014200e+08,api_verified,95,0,"Sapang Palay Proper, San Jose del Monte, Bulac...",14.8388037,121.0426000
1,12 General Espino South Signal,12 GENERAL ESPINO SOUTH SIGNAL,SOUTH DAANG HARI,27.0,CITY OF TAGUIG,7.0,METRO MANILA,76.0,NATIONAL CAPITAL REGION (NCR),13.0,1.381500e+09,api_verified,90,85,"General Espino Street, South Signal Village, S...",14.5062907,121.0536642
2,Vigan ilocos sur,VIGAN CITY ILOCOS SUR,AYUSAN SUR,2.0,CITY OF VIGAN,34.0,ILOCOS SUR,29.0,REGION I (ILOCOS REGION),1.0,1.029340e+08,api_verified,90,0,"United Church of Christ of the Philippines, Ca...",17.5734681,120.3845268



── Invalid (all rows) ──


,original_address,address_normalized,confidence,city_score,province_score,osm_display,api_status
0,19 Luzon Street Batasan Hills 67C Jacson St Qu...,19 LUZON STREET BATASAN HILLS 67C JACSON STREE...,none,0,0,NaN,no_result
1,loilo City,LOILO CITY,none,0,0,NaN,no_result
2,55 Dawn St South Kim View Park SSS Village Con...,55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE...,none,0,0,NaN,no_result
3,carmencdoc Carmen CAGAYAN DE ORO CITY (Capital...,CARMENCDOC CARMEN CAGAYAN DE ORO CITY (CAPITAL...,none,0,0,NaN,no_result


## Cell 12 — One-shot re-run helper

Use this cell to re-run the entire pipeline in one shot after changing config in Cell 1.  
It re-uses all the functions defined above — no need to re-run Cells 3–6 unless you change the reference files.

In [107]:
def run_pipeline(
    input_file   = INPUT_FILE,
    dim_path     = DIM_LOCATION,
    alias_path   = ALIAS_FILE,
    out_verified = OUT_VERIFIED,
    out_invalid  = OUT_INVALID,
    max_rows     = MAX_ROWS,
    use_api      = USE_API,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()
    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(
        dim_path, alias_path
    )
    _re, _rmap = build_alias_regex(_alias)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]
    df['address_normalized'] = df[col].apply(
        lambda x: normalize_alias(str(x), _re, _rmap)
    )

    recs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Fuzzy', unit='row'):
        rec = {
            'original_address': row[col],
            'address_normalized': row['address_normalized'],
            'api_status': 'pending' if use_api else 'skipped',
            'osm_city': None, 'osm_province': None, 'osm_region': None,
            'osm_display': None, 'osm_lat': None, 'osm_lon': None,
        }
        rec.update(match_address(
            row['address_normalized'],
            _dim, _cities, _provinces, _regions, _barangays,
        ))
        recs.append(rec)

    if use_api:
        hi = []
        lo = recs
        log.info(
            f'API mode enabled: queued all rows ({len(lo):,}) for verification '
            f'with query mode={api_query_mode}'
        )
        if lo:
            lo = api_verify_batch(
                lo, _dim, _cities, _provinces, _regions,
                query_mode=api_query_mode,
            )
    else:
        hi = [r for r in recs if not r['needs_api']]
        lo = [r for r in recs if r['needs_api']]
        log.info(f'High-conf: {len(hi):,}  |  Needs API: {len(lo):,}')

    merged = pd.DataFrame(hi + lo)

    if use_api:
        merged['is_valid'] = (
            merged['api_status'].isin(['matched', 'province_only']) &
            (merged['city_name'].notna() | merged['province_name'].notna())
        )
    else:
        merged['is_valid'] = (
            merged['city_name'].notna() |
            (merged['province_name'].notna() & (merged['province_score'] >= PROV_SCORE_HIGH))
        )

    v_df = merged[merged['is_valid']][
        [c for c in VERIFIED_COLS if c in merged.columns]
    ].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][
        [c for c in INVALID_COLS if c in merged.columns]
    ].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid,  'Invalid',  '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df


# Uncomment to run:
# verified_df, invalid_df = run_pipeline()
# verified_df, invalid_df = run_pipeline(api_query_mode='strict')
